<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/Course_Work/RAG_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain_core langchain_groq langchain_text_splitters langchain_community langchain_huggingface chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-http opentelemetry-proto opentelemetry-exporter-otlp-proto-common opentelemetry-semantic-conventions --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0

In [2]:
# Import everything we need
import os
from pathlib import Path
from getpass import getpass

# LangChain core
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.documents import Document

# to access keys
from google.colab import userdata

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
mykey=userdata.get('GROQ_API')
# Set your GROQ API Key
#GROQ_API_KEY = getpass("🔑 Enter your GROQ API Key: ")
os.environ["GROQ_API_KEY"] = mykey
print("✅ API Key set!")

✅ API Key set!


In [4]:
# Create a directory for our policy documents
docs_dir = Path("hr_policies")
docs_dir.mkdir(exist_ok=True)

# ── Policy Document 1: Leave Policy ──────────────────────────────────────────
leave_policy = """
ACME Corp - Leave Policy v2024
Effective Date: January 1, 2024

1. ANNUAL / PRIVILEGE LEAVE
- Full-time employees (permanent): 21 days per calendar year
- Part-time employees: 10.5 days per calendar year (pro-rated)
- Carry forward: Maximum 10 days can be carried forward to the next year
- Encashment: Unused leave beyond 10 days will be encashed at basic salary rate
- Minimum notice: 3 days advance notice required for planned leave
- Approval: Immediate manager must approve; HR records the leave

2. SICK LEAVE
- All employees: 12 days per year
- Medical certificate required for absence exceeding 3 consecutive days
- Sick leave does not carry forward to the next year
- Emergency sick leave can be availed without prior approval
- Employees on probation are entitled to 6 days sick leave

3. MATERNITY LEAVE
- Female employees: 26 weeks (182 days) paid maternity leave
- Applicable after 80 days of employment
- For adoption of a child below 3 months: 12 weeks (84 days)
- Miscarriage or medical termination: 6 weeks leave with full pay

4. PATERNITY LEAVE
- Male employees: 5 working days within 30 days of child's birth
- Fully paid; no carry forward

5. BEREAVEMENT LEAVE
- Immediate family (spouse, child, parent, sibling): 5 days paid
- Extended family (grandparents, in-laws): 3 days paid
- Employee must inform HR and manager as soon as possible

6. COMPENSATORY OFF (COMP-OFF)
- Employees who work on public holidays or weekends are entitled to a comp-off
- Comp-off must be availed within 60 days of the working day
- Maximum comp-off balance: 5 days at any time
- Approval from manager required before working on a holiday

7. PUBLIC HOLIDAYS
- National holidays: 3 days (Republic Day, Independence Day, Gandhi Jayanti)
- Festival holidays: 8 days (as per location-specific calendar)
- Optional holidays: 2 days (employee's choice from the approved list)

Contact: hr@acme.com | Policy Owner: Head of HR
"""

# ── Policy Document 2: Reimbursement & Expense Policy ─────────────────────────
expense_policy = """
ACME Corp - Travel & Expense Reimbursement Policy v2024
Effective Date: January 1, 2024

1. BUSINESS TRAVEL
a) Domestic Travel
   - Flight: Economy class for flights under 4 hours; Business class for 4+ hours with VP approval
   - Train: AC First Class or AC 2-Tier permitted
   - Hotel: Up to INR 5,000 per night in Tier-1 cities; INR 3,500 in Tier-2/Tier-3 cities
   - Daily Allowance (DA): INR 600/day for Tier-1 cities; INR 400/day for others
   - Local transport: Actual cost with receipt; cab booking via approved corporate account preferred

b) International Travel
   - All international travel requires Director-level approval minimum
   - Business class permitted for flights 8+ hours
   - Hotel: As per country-specific guidelines; budget cap applies
   - Forex expenses: Supported through company corporate card; personal card reimbursed at RBI rate

2. REIMBURSEMENT PROCESS
   - Submit claims within 15 days of expense incurrence
   - Upload original bills/receipts on the expense portal (portal.acme.com/expenses)
   - Claims submitted after 30 days will not be reimbursed without CFO approval
   - Manager approval required for all claims
   - Reimbursement processed within 7 working days of approval

3. MEAL REIMBURSEMENT
   - Working meals with clients: Up to INR 2,500 per person with manager approval
   - Team meals (10+ people): Up to INR 1,500 per person with VP approval
   - Alcohol: NOT reimbursable

4. MOBILE & INTERNET
   - Mobile reimbursement: INR 1,000/month for employees with field-work roles
   - Internet at home (WFH): INR 750/month for employees on WFH scheme
   - Claims must be submitted monthly with bill

5. TRAINING & CERTIFICATION
   - Job-relevant certification: 100% reimbursement up to INR 25,000 per year
   - Employee must complete the certification (no reimbursement for failed attempts)
   - Bond: 1-year service commitment post-reimbursement for amounts above INR 15,000
   - Pre-approval from manager and HR required before enrollment

Contact: finance@acme.com | Policy Owner: CFO Office
"""

# ── Policy Document 3: Code of Conduct ────────────────────────────────────────
code_of_conduct = """
ACME Corp - Code of Conduct and Workplace Policy v2024
Effective Date: January 1, 2024

1. WORKING HOURS
   - Standard hours: 9:00 AM to 6:00 PM, Monday to Friday
   - Core hours: Employees must be available 10:00 AM to 4:00 PM
   - Flexible hours: Employees may adjust start/end within 8:00 AM - 7:00 PM window
   - Overtime: Eligible employees compensated at 1.5x rate for hours beyond 9 hours/day

2. WORK FROM HOME (WFH) POLICY
   - Employees with 6+ months tenure can apply for hybrid WFH (up to 2 days/week)
   - Employees with 3+ years tenure eligible for 3 days/week WFH with manager approval
   - Full-time WFH requires VP-level approval and is reviewed every 6 months
   - WFH not applicable during probation period (first 3 months)
   - Employees must be available on video call during core hours while WFH

3. DRESS CODE
   - Office: Business casual from Monday to Thursday; casual on Fridays
   - Client meetings: Formal attire required
   - No torn clothing, offensive graphics, or beachwear in office

4. ANTI-HARASSMENT POLICY
   - Zero tolerance for any form of harassment — sexual, verbal, physical, or online
   - Report incidents to HR at posh@acme.com or call the POSH helpline: 1800-XXX-XXXX
   - All complaints are investigated within 30 days
   - Retaliation against complainants is a serious violation leading to termination

5. CONFIDENTIALITY
   - Employees must not disclose company IP, client data, or financials
   - NDA must be signed at onboarding; survives employment for 2 years post-exit
   - Social media: Employees must not post confidential company information online

6. PERFORMANCE REVIEW
   - Annual performance review cycle: January–March each year
   - Mid-year check-in: July for goal alignment
   - Ratings: Exceeds Expectations (5), Meets Expectations (3-4), Needs Improvement (1-2)
   - Salary revision effective April 1st; linked to performance rating
   - Employees rated 2 or below enter Performance Improvement Plan (PIP) — 90 days

7. GRIEVANCE REDRESSAL
   - Step 1: Raise with immediate manager
   - Step 2: Escalate to HR Business Partner if unresolved in 5 working days
   - Step 3: Final escalation to Head of HR
   - Anonymous grievance submission: grievance@acme.com

Contact: hr@acme.com | Policy Owner: Head of HR
"""

# Write files to disk
(docs_dir / "leave_policy.txt").write_text(leave_policy)
(docs_dir / "expense_policy.txt").write_text(expense_policy)
(docs_dir / "code_of_conduct.txt").write_text(code_of_conduct)

print("✅ Created 3 HR policy documents:")
for f in docs_dir.iterdir():
    print(f"   📄 {f.name} ({f.stat().st_size} bytes)")

✅ Created 3 HR policy documents:
   📄 leave_policy.txt (1922 bytes)
   📄 code_of_conduct.txt (2282 bytes)
   📄 expense_policy.txt (2045 bytes)


In [5]:
# ── Step 3a: Load Documents ───────────────────────────────────────────────────
print("📥 Loading documents...")

loader = DirectoryLoader(
    "hr_policies",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
raw_docs = loader.load()

print(f"✅ Loaded {len(raw_docs)} documents")
for doc in raw_docs:
    source = doc.metadata.get('source', 'unknown')
    print(f"   📄 {Path(source).name} — {len(doc.page_content)} characters")

📥 Loading documents...
✅ Loaded 3 documents
   📄 leave_policy.txt — 1922 characters
   📄 code_of_conduct.txt — 2276 characters
   📄 expense_policy.txt — 2045 characters


In [6]:
# ── Step 3b: Split into Chunks ────────────────────────────────────────────────
print("✂️  Splitting documents into chunks...")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Max characters per chunk
    chunk_overlap=100,     # Overlap between chunks (for context continuity)
    separators=["\n\n", "\n", ".", " "],  # Split order preference
    length_function=len
)

chunks = splitter.split_documents(raw_docs)

print(f"✅ Created {len(chunks)} chunks from {len(raw_docs)} documents")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

# 🔍 Inspect a sample chunk
print("\n🔍 Sample Chunk #5:")
print("-" * 50)
print(chunks[5].page_content)
print("-" * 50)
print(f"Source: {Path(chunks[5].metadata['source']).name}")

✂️  Splitting documents into chunks...
✅ Created 19 chunks from 3 documents
   Average chunk size: 326 characters

🔍 Sample Chunk #5:
--------------------------------------------------
7. PUBLIC HOLIDAYS
- National holidays: 3 days (Republic Day, Independence Day, Gandhi Jayanti)
- Festival holidays: 8 days (as per location-specific calendar)
- Optional holidays: 2 days (employee's choice from the approved list)

Contact: hr@acme.com | Policy Owner: Head of HR
--------------------------------------------------
Source: leave_policy.txt


In [7]:
# ── Step 3c: Create Embeddings ────────────────────────────────────────────────
# Using HuggingFace's free sentence-transformers model
# ✅ No API key needed for embeddings!

print("🔢 Loading embedding model (this may take a moment on first run)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}  # Cosine similarity optimization
)

# Test the embedding model
test_embedding = embeddings.embed_query("What is the sick leave policy?")
print("✅ Embedding model loaded!")
print(f"   Embedding dimensions: {len(test_embedding)}")
print(f"   Sample values: {test_embedding[:5]}")

🔢 Loading embedding model (this may take a moment on first run)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
   Embedding dimensions: 384
   Sample values: [0.04612019658088684, 0.06519241631031036, -0.0278937928378582, 0.027578696608543396, 0.1055070236325264]


In [8]:
# ── Step 3d: Store in ChromaDB ────────────────────────────────────────────────
print("🗄️  Building ChromaDB vector store...")

# This creates embeddings for ALL chunks and stores them in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",  # Persist to disk
    collection_name="hr_policies"
)

print(f"✅ Vector store created with {vectorstore._collection.count()} vectors!")

# Quick test: Similarity search
print("\n🔍 Testing retrieval — Query: 'sick leave days' ")
test_results = vectorstore.similarity_search("sick leave days", k=2)
for i, r in enumerate(test_results):
    print(f"\nResult {i+1} (from {Path(r.metadata['source']).name}):")
    print(r.page_content[:200] + "...")

🗄️  Building ChromaDB vector store...
✅ Vector store created with 19 vectors!

🔍 Testing retrieval — Query: 'sick leave days' 

Result 1 (from leave_policy.txt):
2. SICK LEAVE
- All employees: 12 days per year
- Medical certificate required for absence exceeding 3 consecutive days
- Sick leave does not carry forward to the next year
- Emergency sick leave can ...

Result 2 (from leave_policy.txt):
ACME Corp - Leave Policy v2024
Effective Date: January 1, 2024...


In [9]:
# Initialize the GROQ LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # Latest LLaMA 3.3 70B
    temperature=0.1,                    # Low temp = more factual, less creative
    max_tokens=1024,
    groq_api_key=mykey
)

# Test the LLM directly (NO RAG yet)
print("🤖 Testing GROQ LLM directly (WITHOUT any policy context):")
print("-" * 60)

test_query = "What is the sick leave policy at ACME Corp?"
response = llm.invoke(test_query)

print(f"Q: {test_query}")
print("\nA (Base LLM — may hallucinate!):")
print(response.content)

print("\n⚠️  Notice: The LLM doesn't know ACME Corp's specific policies!")
print("   It either says it doesn't know OR makes up plausible-sounding policies.")

🤖 Testing GROQ LLM directly (WITHOUT any policy context):
------------------------------------------------------------
Q: What is the sick leave policy at ACME Corp?

A (Base LLM — may hallucinate!):
I don't have access to specific information about ACME Corp's policies, including their sick leave policy. ACME Corp is often used as a fictional company in examples and illustrations, so it's possible that it may not be a real company. If you're looking for information about a specific company's sick leave policy, I recommend checking their official website, employee handbook, or contacting their HR department directly. They should be able to provide you with the most up-to-date and accurate information.

⚠️  Notice: The LLM doesn't know ACME Corp's specific policies!
   It either says it doesn't know OR makes up plausible-sounding policies.


In [10]:
# ── 1. Prompt Template ────────────────────────────────────────────────────────
# ChatPromptTemplate is the modern way — works with chat models (system + human)
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a helpful HR assistant for ACME Corp.
Answer the employee's question using ONLY the information in the provided context.

Rules:
- Be precise and factual; cite specific numbers and dates from the policy.
- If the answer is not in the context, say: "I don't have this information in the current HR policies. Please contact hr@acme.com"
- Do NOT make up policies or numbers that aren't in the context.
- Keep answers concise but complete.

Context from HR Policy Documents:
──────────────────────────────────
{context}
──────────────────────────────────"""),
    ("human", "{question}")
])


# ── 2. Helper: format retrieved docs into a single string ─────────────────────
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {Path(doc.metadata['source']).name}]\n{doc.page_content}"
        for doc in docs
    )

# ── Create the Retriever ──────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Cosine similarity search
    search_kwargs={
        "k": 4  # Return top 4 most relevant chunks
    }
)

# ── 3. LCEL RAG Chain ─────────────────────────────────────────────────────────
#
# RunnableParallel runs TWO branches simultaneously:
#   • "context"  → retriever fetches relevant chunks → format_docs converts to string
#   • "question" → RunnablePassthrough() passes the original question unchanged
# Both outputs flow into the prompt template, then the LLM, then the string parser.

rag_chain = (
    RunnableParallel({
        "context":  retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    })
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ LCEL RAG chain built!")
print()
print("Chain breakdown:")
print("  RunnableParallel(context=retriever|format_docs, question=passthrough)")
print("  | ChatPromptTemplate  (injects context + question into prompt)")
print("  | ChatGroq            (LLaMA 3.3 70B via GROQ API)")
print("  | StrOutputParser     (extracts plain text from AIMessage)")

✅ LCEL RAG chain built!

Chain breakdown:
  RunnableParallel(context=retriever|format_docs, question=passthrough)
  | ChatPromptTemplate  (injects context + question into prompt)
  | ChatGroq            (LLaMA 3.3 70B via GROQ API)
  | StrOutputParser     (extracts plain text from AIMessage)


In [11]:
def ask(question: str) -> str:
    """Invoke the LCEL RAG chain."""
    return rag_chain.invoke(question)

questions = [
    "How many sick leave days do I get per year?",
    "Can I carry forward unused annual leave?",
    "What is the hotel reimbursement limit in Mumbai?",
]

for q in questions:
    print(f"\n{'='*65}")
    print(f"❓ {q}")
    print(f"🤖 {ask(q)}")


❓ How many sick leave days do I get per year?
🤖 You are entitled to 12 days of sick leave per year, unless you are on probation, in which case you are entitled to 6 days.

❓ Can I carry forward unused annual leave?
🤖 Yes, you can carry forward a maximum of 10 days of unused annual leave to the next year.

❓ What is the hotel reimbursement limit in Mumbai?
🤖 Mumbai is considered a Tier-1 city. The hotel reimbursement limit in Tier-1 cities is up to INR 5,000 per night.


In [12]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
cmp_q = "What happens if I submit an expense claim after 30 days?"

print("🔬 SIDE-BY-SIDE: Base LLM vs RAG")
print("="*65)
print(f"Question: {cmp_q}\n")

print("❌ Base LLM (no context):")
print(llm.invoke(cmp_q).content)

print("\n✅ RAG ANSWER (policy-grounded):")
print(ask(cmp_q))

print("\n💡 RAG gives ACME Corp's exact policy; Base LLM can only guess.")

🔬 SIDE-BY-SIDE: Base LLM vs RAG
Question: What happens if I submit an expense claim after 30 days?

❌ Base LLM (no context):
The consequences of submitting an expense claim after 30 days can vary depending on the organization's policies and procedures. Here are some possible outcomes:

1. **Rejection**: The claim might be rejected outright, and you may not be reimbursed for the expenses.
2. **Late submission fee**: You might be charged a late submission fee or a penalty for not submitting the claim within the specified timeframe.
3. **Reduced reimbursement**: The organization might reduce the amount of reimbursement or apply a penalty to the claim.
4. **Managerial approval required**: The claim might require additional approval from a manager or supervisor, which could delay the reimbursement process.
5. **Tax implications**: If the expense claim is related to a tax-deductible expense, submitting it late might affect your tax obligations or eligibility for tax deductions.
6. **Impact o